# Notebook 1: Data Loading & Cleaning

## Dataset & Scope

This project originally proposed analyzing all ~5.7 million federal contract
awards in FY25. That full dataset spans four contract subtypes available on
USAspending.gov:

| Contract Type | Row Count |
|---|---|
| Delivery Orders | ~4,193,809 |
| BPA Calls | ~778,846 |
| Purchase Orders | ~704,164 |
| **Definitive Contracts** | **~73,365** |

After attempting to download the full dataset, the zip file generation on
USAspending.gov failed repeatedly, which is a known infrastructure limitation of the
platform for large bulk exports. The USAspending API was also considered as an
alternative extraction method, but its documentation is sparse and the
pagination required to retrieve 5.7 million records would have been
too time-consuming.

### Why Definitive Contracts?

Rather than treating the scope reduction as a limitation or failure, using Definitive Contracts allows for the best insights, as they
are arguably the most analytically meaningful subtype for this project's
research questions. A Definitive Contract is a fully negotiated, legally binding
agreement between a federal agency and a contractor for a fixed scope of work at
an agreed price. This makes them the clearest signal of deliberate, strategic
procurement decisions.

The excluded subtypes each carry analytical noise that would complicate network
analysis:

- **Delivery Orders (~4.19M rows):** Task orders issued against pre-existing
  Indefinite Delivery contracts. They represent execution of previously
  negotiated agreements rather than new procurement decisions, and their sheer
  volume would require cloud infrastructure to process reliably.
- **BPA Calls (~778K rows):** Simplified purchases against Blanket Purchase
  Agreements — essentially recurring supply orders. They reflect operational
  purchasing rather than strategic contracting relationships.
- **Purchase Orders (~704K rows):** Small, often one-time transactions typically
  below simplified acquisition thresholds. They tend to be low-value and
  administratively routine, adding volume without meaningful network signal.

Definitive Contracts represent the federal government's most deliberate and
consequential procurement decisions. These are the contracts that define long-term
agency-contractor relationships, involve formal competition, and carry the
largest strategic weight. For the purpose of network analysis, HITS scoring,
and contractor profiling, this is the highest-signal subset available.

**Data source:** USAspending.gov — FY2025 Definitive Contracts, Prime Award Summaries  
**Input:** `Contracts_PrimeAwardSummaries.csv` (73,365 rows × 286 columns)  
**Output:** `contracts_cleaned.csv` (73,350 rows × 6 columns)

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 50)

In [27]:
df = pd.read_csv('Contracts_PrimeAwardSummaries.csv', low_memory=False)

print(f"Shape: {df.shape}")
print(f"\nColumn names:\n{list(df.columns)}")

Shape: (73365, 286)

Column names:
['contract_award_unique_key', 'award_id_piid', 'parent_award_agency_id', 'parent_award_agency_name', 'parent_award_id_piid', 'disaster_emergency_fund_codes', 'outlayed_amount_from_COVID-19_supplementals', 'obligated_amount_from_COVID-19_supplementals', 'outlayed_amount_from_IIJA_supplemental', 'obligated_amount_from_IIJA_supplemental', 'total_obligated_amount', 'total_outlayed_amount', 'current_total_value_of_award', 'potential_total_value_of_award', 'award_base_action_date', 'award_base_action_date_fiscal_year', 'award_latest_action_date', 'award_latest_action_date_fiscal_year', 'period_of_performance_start_date', 'period_of_performance_current_end_date', 'period_of_performance_potential_end_date', 'ordering_period_end_date', 'solicitation_date', 'awarding_agency_code', 'awarding_agency_name', 'awarding_sub_agency_code', 'awarding_sub_agency_name', 'awarding_office_code', 'awarding_office_name', 'funding_agency_code', 'funding_agency_name', 'funding_

## Column Selection

The raw download contains 286 columns covering contractor demographics,
competition details, financial breakdowns, socioeconomic certifications, and
more. Only 6 columns are analytically relevant to this project's pipeline:

| Column | Purpose |
|---|---|
| `contract_award_unique_key` | Unique row identifier |
| `recipient_name` | Contractor name — MinHash input, HITS node, clustering subject |
| `recipient_uei` | Government-assigned unique entity identifier — used for MinHash validation |
| `awarding_agency_name` | Federal agency — HITS hub node |
| `naics_code` | Service classification — Apriori items |
| `total_obligated_amount` | Contract value — clustering feature |

In [28]:
COLS = [
    'contract_award_unique_key',
    'recipient_name',
    'recipient_uei',
    'awarding_agency_name',
    'naics_code',
    'total_obligated_amount',
]

df = df[COLS]
print(f"Trimmed shape: {df.shape}")
df.head()

Trimmed shape: (73365, 6)


,contract_award_unique_key,recipient_name,recipient_uei,awarding_agency_name,naics_code,total_obligated_amount
0,CONT_AWD_36C10F25C0001_3600_-NONE-_-NONE-,"S. J. AMOROSO CONSTRUCTION CO., LLC",KU19CNLCK2K7,Department of Veterans Affairs,236220.0,69517876.07
1,CONT_AWD_36C10F25C0002_3600_-NONE-_-NONE-,"SPUR DESIGN, LLC",FLGHXSNF1NX5,Department of Veterans Affairs,541330.0,2919358.00
2,CONT_AWD_36C10F25C0004_3600_-NONE-_-NONE-,"3LINKS TECHNOLOGIES, INC.",GKDSP7FWELX6,Department of Veterans Affairs,238210.0,1848640.00
3,CONT_AWD_36C10F25C0005_3600_-NONE-_-NONE-,RANDY KINDER EXCAVATING INC,XMKUEVF3NMB5,Department of Veterans Affairs,238910.0,3677824.00
4,CONT_AWD_36C10F25C0007_3600_-NONE-_-NONE-,HAMILTON PACIFIC CHAMBERLAIN LLC,EDXHGN4LK5T8,Department of Veterans Affairs,236220.0,639183.95


In [29]:
## Null Analysis
print("=== Null Counts ===")
print(df.isnull().sum())

print("\n=== % Null ===")
print((df.isnull().sum() / len(df) * 100).round(2))

=== Null Counts ===
contract_award_unique_key     0
recipient_name                0
recipient_uei                 0
awarding_agency_name          0
naics_code                   15
total_obligated_amount        0
dtype: int64

=== % Null ===
contract_award_unique_key    0.00
recipient_name               0.00
recipient_uei                0.00
awarding_agency_name         0.00
naics_code                   0.02
total_obligated_amount       0.00
dtype: float64


## Data Quality Assessment

Null counts and basic statistics are assessed before cleaning to understand
the scope of any data quality issues.

In [30]:
## Basic Statistics
print("=== Obligated Amount Stats ===")
print(df['total_obligated_amount'].describe())

print("\n=== Unique Counts ===")
print(f"Unique contractors (recipient_name): {df['recipient_name'].nunique()}")
print(f"Unique UEIs: {df['recipient_uei'].nunique()}")
print(f"Unique agencies: {df['awarding_agency_name'].nunique()}")
print(f"Unique NAICS codes: {df['naics_code'].nunique()}")

=== Obligated Amount Stats ===
count    7.336500e+04
mean     3.913169e+07
std      6.618580e+08
min     -2.024625e+06
25%      1.935083e+05
50%      8.769425e+05
75%      3.470143e+06
max      5.126921e+10
Name: total_obligated_amount, dtype: float64

=== Unique Counts ===
Unique contractors (recipient_name): 24807
Unique UEIs: 25540
Unique agencies: 55
Unique NAICS codes: 787


## Cleaning & Stage 1 Entity Normalization

Three cleaning steps are applied:

1. **Drop missing NAICS codes**: 15 rows lack a NAICS code, consistent with
   non-North American contractors whose business activity falls outside the
   North American Industry Classification System scope. These rows are dropped
   as NAICS codes are the foundation of the Apriori analysis in Notebook 4.

2. **Standardize text fields**: Agency and contractor names are uppercased
   and stripped of leading/trailing whitespace.

3. **Stage 1 entity normalization**: Contractor names are normalized by
   replacing `&` with `AND`, removing all punctuation, and collapsing
   whitespace. This deterministic cleaning step resolves exact duplicates
   (e.g. `BOOZ ALLEN HAMILTON INC.` → `BOOZ ALLEN HAMILTON INC`) before
   probabilistic MinHash resolution in Notebook 2, reducing the unique name
   count and improving downstream accuracy.

In [31]:
# Drop rows missing NAICS code
df = df.dropna(subset=['naics_code'])
print(f"Rows after dropping missing NAICS: {len(df)}")

# Standardize text fields
df['awarding_agency_name'] = df['awarding_agency_name'].str.strip()
df['recipient_name'] = df['recipient_name'].str.upper().str.strip()

# Convert NAICS to string
df['naics_code'] = df['naics_code'].astype(int).astype(str)

# Normalize recipient names for exact deduplication
# Replace & with AND, remove punctuation, collapse spaces
def normalize_name(name):
    name = name.replace('&', 'AND')
    name = re.sub(r'[^A-Z0-9 ]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

df['recipient_name'] = df['recipient_name'].apply(normalize_name)

print(f"Final shape: {df.shape}")
print(f"Unique contractor names after normalization: {df['recipient_name'].nunique():,}")

Rows after dropping missing NAICS: 73350
Final shape: (73350, 6)
Unique contractor names after normalization: 24,618


## Save

Cleaned dataset is saved as `contracts_cleaned.csv` for use in Notebook 2.

In [32]:
df.to_csv('contracts_cleaned.csv', index=False)
print("Saved contracts_cleaned.csv")
print(f"Clean Dataset: {df.shape[0]} rows × {df.shape[1]} columns")

Saved contracts_cleaned.csv
Clean Dataset: 73350 rows × 6 columns
